# Terminology
* bounds vs. positions (begin, end, location)
* **chains**
* **relation** vs. conflation vs. merge vs. join
* **overlay** vs. **intersect** vs. overlap vs. ?
* integrate vs. unify vs. combine vs. interweave vs. intertwine
# Desired Features
* Segmentize events
* Project parallel lines
* Project polygons

# Commonly Used Features
* Union - e.g., unifying HIN and HRN
* HIN - resegmentation, conflation, and distribution; including variable lengths for resegmentation
* Event aggregation - e.g., counting crashes on segments
* generate_linear_events - e.g., received data doesn't have LRS but is needed for HIN or other
* defining intersection influence areas

# Dependencies and Data

In [1]:
import pandas as pd
import geopandas as gpd
import linref as lr

In [2]:
# Set default Linear Referencing System (LRS)
LRS =   lr.LRS(key_col=['NLFID', 'chain'], loc_col='COUNTY_LOG_NBR', beg_col='CTL_BEGIN_', end_col='CTL_END_NB', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')
lr.set_default_lrs(LRS)

In [3]:
# Load linear events with geometry
roads =   gpd.read_file('testing/data/roadways.json').explode().rename(columns={'NLF_ID': 'NLFID'})
crashes = gpd.read_file('testing/data/crashes.json')

Skipping field INTERSECTION_LEG_ID: unsupported OGR type: 5


In [4]:
roads.lr

LRS_Accessor with linear referencing system (LRS):
[gr lc LN SP sm] LRS(key_col=['NLFID', 'chain'], loc_col='COUNTY_LOG_NBR', beg_col='CTL_BEGIN_', end_col='CTL_END_NB', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')

# Preparing LRS Data

In [5]:
# Add M values and chaining to roadways
roads = roads.lr.add_geom_m()
roads = roads.lr.add_chaining(replace=True)
# Impute the chaining keys to the crash data
crashes = roads.lr.impute_keys(crashes)
# Filter out crashes that don't have complete LRS data
crashes = crashes[crashes.lr.valid_events]

/home/tshihadah/code/linref/linref/ext/base.py:1468: UserWarning: 62 rows contain missing key values after imputation.
  warnings.warn(msg, UserWarning)


# Conflating Attributes

In [ ]:
# Create relationship between roads and crashes
relation = roads.lr.relate(crashes)
# Use a variety of aggregation methods to conflate crash years onto roadways
# - Multiple attributes can be aggregated at once by passing a list of columns
# - Certain aggregation methods require specific data types
# - Aggregation methods may operate differently for point vs. linear events
# - Some aggregation methods can use either a count-based or length-based weighting scheme
roads['CRASH_YR_FIRST'] = relation['CRASH_YR'].first()
roads['CRASH_YR_LAST']  = relation['CRASH_YR'].last()
roads['CRASH_YR_LIST']  = relation['CRASH_YR'].list()
roads['CRASH_YR_SET']   = relation['CRASH_YR'].set()
roads['CRASH_YR_SUM']   = relation['CRASH_YR'].sum()
roads['CRASH_YR_MEAN']  = relation['CRASH_YR'].mean()
roads['CRASH_YR_MODE']  = relation['CRASH_YR'].mode()
roads['CRASH_YR_COUNT'] = relation['CRASH_YR'].count()
# View a sample of the results
roads[['CRASH_YR_FIRST', 'CRASH_YR_LAST', 'CRASH_YR_LIST', 'CRASH_YR_SET', 'CRASH_YR_SUM', 'CRASH_YR_MEAN', 'CRASH_YR_MODE', 'CRASH_YR_COUNT']].sample(10)

,CRASH_YR_FIRST,CRASH_YR_LAST,CRASH_YR_LIST,CRASH_YR_SET,CRASH_YR_SUM,CRASH_YR_MEAN,CRASH_YR_MODE,CRASH_YR_COUNT
1329,NaN,NaN,[],{},0.0,0.0,NaN,0.0
3044,NaN,NaN,[],{},0.0,0.0,NaN,0.0
1512,NaN,NaN,[],{},0.0,0.0,NaN,0.0
3620,NaN,NaN,[],{},0.0,0.0,NaN,0.0
4982,NaN,NaN,[],{},0.0,0.0,NaN,0.0
4918,NaN,NaN,[],{},0.0,0.0,NaN,0.0
1719,NaN,NaN,[],{},0.0,0.0,NaN,0.0
1037,NaN,NaN,[],{},0.0,0.0,NaN,0.0
355,2019.0,2019.0,[2019],{2019},2019.0,2019.0,2019.0,1.0
760,NaN,NaN,[],{},0.0,0.0,NaN,0.0


# Dissolving Linear Data

In [35]:
# Perform dissolve
dissolved = roads.lr.dissolve()

# Resegmenting Linear Data at Regular Intervals

In [36]:
# Resegment the dissolved data at 0.5 mile intervals
segments = dissolved.lr.resegment(length=0.5, fill='balance')

# Integrate Multiple Sets of Linear Events

In [38]:
# Create additional resegmented linear events for example
segments_1 = dissolved.lr.resegment(length=0.5, fill='balance').sample(frac=0.5, random_state=1)
segments_2 = dissolved.lr.resegment(length=0.3, fill='balance').sample(frac=0.5, random_state=1)
# Integrate the resegmented linear events
integrated = lr.integrate([segments_1, segments_2])
# Cut new geometry at integrated locations
integrated = integrated.lr.cut_from(dissolved)

# Project Point Events onto the LRS

In [39]:
# Project crash points onto the dissolved linear events
projected = dissolved.lr.project(crashes.drop(columns=['COUNTY_LOG_NBR', 'NLFID', 'chain']))
# Interpolate new geometry at projected locations along roadway events
projected = projected.lr.interpolate_from(dissolved)